# YouTube-bot One-Tap Mobile Video Builder

Open on a phone, tap **Runtime → Run all**, and wait for the ZIP download. The notebook uses a fixed tested commit and a clean absolute `/content` workspace so reruns do not create nested folders.

Default output: offline `slides` MP4. No keys, computer, GitHub token, or YouTube credentials required.

In [ ]:
#@title Set topic and output
TOPIC = 'Why the sky changes color' #@param {type:'string'}
EPISODE_ID = 'mobile-one-tap-001' #@param {type:'string'}
VIDEO_MODE = 'slides' #@param ['slides', 'ken-burns', 'shorts-vertical', 'audiogram']
PINNED_COMMIT = '63a4d57d4a90a50707044447fe810cbaec4f2f41'
print(f'Topic: {TOPIC} | Mode: {VIDEO_MODE}')

In [ ]:
#@title Build, recover, and download
import os, shutil, subprocess, glob
from pathlib import Path
from google.colab import files
work = Path('/content/yt-agent-mobile')
if work.exists(): shutil.rmtree(work)
subprocess.run(['git','clone','-q','https://github.com/studio1111/YouTube-bot.git',str(work / 'repo')], check=True)
repo = work / 'repo'
subprocess.run(['git','-C',str(repo),'checkout','-q',PINNED_COMMIT], check=True)
subprocess.run(['python','-m','pip','install','-q','-e','.'], cwd=repo, check=True)
if shutil.which('ffmpeg') is None:
    subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y ffmpeg'], check=True)
run = subprocess.run(['python','-m','yt_agent.cli','run','--topic',TOPIC,'--episode-id',EPISODE_ID,'--runtime-dir','runtime','--video-mode',VIDEO_MODE], cwd=repo, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0: print(run.stderr)
episode = repo / 'runtime' / 'episodes' / EPISODE_ID / 'v1'
video = episode / 'video.mp4'
# The pinned commit may leave video-not-rendered.txt; recover directly from its frames.if not video.exists():
    frames = episode / 'frames'
    audio = episode / 'narration.wav'
    ppm = sorted(frames.glob('slide-*.ppm'))
    if ppm and audio.exists() and shutil.which('ffmpeg'):
        recover = subprocess.run(['ffmpeg','-y','-nostdin','-loglevel','error','-framerate','1','-i',str(frames/'slide-%03d.ppm'),'-i',str(audio),'-vf','scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2,format=yuv420p','-c:v','libx264','-c:a','aac','-shortest',str(video)], text=True, capture_output=True)
        if recover.returncode != 0: print('Recovery FFmpeg error:\n' + recover.stderr)
    if not video.exists():
        candidates = [str(p) for p in repo.rglob('video-not-rendered.txt')]
        print('MP4 was not created. Candidate diagnostics:', candidates)
        raise RuntimeError('Video render failed. Use VIDEO_MODE=slides and rerun this cell.')
bundle = work / f'{EPISODE_ID}-video-package'
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(episode, bundle)
archive = shutil.make_archive(str(bundle), 'zip', root_dir=bundle)
files.download(archive)
print(f'Downloaded: {archive}')